<a href="https://colab.research.google.com/github/engMohamedAbdAlslam/DRP_segmentation/blob/copilot%2Fdevelop-preprocessing-pipeline/notebooks/01_disease_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Fundus Image Preprocessing (DDR Dataset)
**Dataset:** DDR Diabetic Retinopathy (`ryanthao/fgadr`) — 12,522 fundus images, DR grading labels
**Goal:** Download via KaggleHub, preprocess fundus images (CLAHE + resize + normalize), save as `.npz`, visualize samples.
> Run on **Google Colab** with T4 GPU. Set your Kaggle credentials in Colab Secrets before running.

## 1. Colab Repo Setup

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/DRP_segmentation')
if not repo_path.exists():
    !git clone https://github.com/engMohamedAbdAlslam/DRP_segmentation.git /content/DRP_segmentation

%cd /content/DRP_segmentation
!git checkout copilot/develop-preprocessing-pipeline
!git pull origin copilot/develop-preprocessing-pipeline
print('Repo ready at', Path.cwd())

## 2. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'kagglehub', 'opencv-python-headless', 'tqdm', 'matplotlib', 'scikit-learn', 'numpy', 'pandas'],
               check=True)
print('Dependencies ready.')

## 3. Kaggle Authentication

In [ ]:
from google.colab import userdata
import os

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(userdata.get('KAGGLE_TOKEN'))
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
print('Kaggle token ready!')

## 4. Repository Setup & Imports

In [ ]:
import sys
import shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import sklearn
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print('numpy', np.__version__)
print('opencv', cv2.__version__)
print('sklearn', sklearn.__version__)

repo_root = Path('/content/DRP_segmentation')
if not (repo_root / 'src').exists():
    raise FileNotFoundError('src/ not found. Make sure Cell 1 ran successfully.')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image, save_preprocessed

data_dir   = repo_root / 'data'
raw_dir    = data_dir / 'raw'
processed_dir = data_dir / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print('Repo root:', repo_root)

## 5. Download DDR Dataset via KaggleHub

In [ ]:
import os
from google.colab import userdata
import kagglehub

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_TOKEN')

DATASET_SLUG = 'ryanthao/fgadr'
DATASET_NAME = 'ddr'               # used for processed output folder
ddr_root = raw_dir / 'ddr'
ddr_root.mkdir(parents=True, exist_ok=True)

already_downloaded = any(ddr_root.rglob('*.jpg')) or any(ddr_root.rglob('*.png'))
if not already_downloaded:
    print(f'Downloading {DATASET_SLUG} ...')
    download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    if download_path != ddr_root:
        shutil.copytree(download_path, ddr_root, dirs_exist_ok=True)
    print('Download complete:', ddr_root)
else:
    print('Dataset already present at:', ddr_root)

print(f'Total files: {len(list(ddr_root.rglob("*")))}') 

## 6. Index Images & Build DataFrame

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

# DDR has no pixel-level lesion masks — classification labels only
all_imgs = sorted([p for p in ddr_root.rglob('*') if p.suffix.lower() in IMAGE_EXTS])

# Try to attach DR grade labels from CSV if available
csv_path = ddr_root / 'DR_grading.csv'
if csv_path.exists():
    label_df = pd.read_csv(csv_path, header=None, names=['filename', 'grade'])
    label_map = dict(zip(label_df['filename'], label_df['grade']))
    rows = [{'image_path': img,
             'mask_paths': [],
             'dr_grade': label_map.get(img.name, -1)} for img in all_imgs]
    print('Labels loaded from CSV.')
else:
    rows = [{'image_path': img, 'mask_paths': [], 'dr_grade': -1} for img in all_imgs]
    print('No CSV found — dr_grade set to -1.')

df = pd.DataFrame(rows)
print(f'Total images: {len(df)}')
print(df['dr_grade'].value_counts().sort_index().rename('count'))
df.head()

## 7. Train / Val / Test Split (70 / 15 / 15)

In [ ]:
if df.empty:
    raise RuntimeError('No images found — check dataset download path.')

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['dr_grade'] if df['dr_grade'].ne(-1).all() else None)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
splits = {'train': train_df, 'val': val_df, 'test': test_df}

## 8. Batch Preprocessing & Save as .npz

In [ ]:
config = PreprocessConfig(target_size=(512, 512), normalization='zero_one')
errors = []

for split_name, split_df in splits.items():
    out_dir = processed_dir / DATASET_NAME / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\nProcessing {split_name} ({len(split_df)} images)...')
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        try:
            result = preprocess_fundus_image(row['image_path'], mask=None, config=config)
            out_file = out_dir / (row['image_path'].stem + '.npz')
            save_preprocessed(result, out_file)
        except Exception as e:
            errors.append({'file': str(row['image_path']), 'error': str(e)})
            print(f'  WARNING: skipped {row["image_path"].name} — {e}')

print(f'\nDone. Total errors: {len(errors)}')
if errors:
    print(pd.DataFrame(errors))

## 9. Dataset Statistics

In [ ]:
print('=== Processed .npz Counts ===')
for split_name in splits:
    count = len(list((processed_dir / DATASET_NAME / split_name).rglob('*.npz')))
    print(f'  {split_name}: {count} files')

# Original image size stats
sample_imgs = list(ddr_root.rglob('*.jpg'))[:500]
widths, heights = [], []
for p in sample_imgs:
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w); heights.append(h)

if widths:
    print(f'\nOriginal Image Sizes (sample of {len(widths)}):')
    print(f'  Width  — min:{min(widths)} max:{max(widths)} mean:{int(np.mean(widths))}')
    print(f'  Height — min:{min(heights)} max:{max(heights)} mean:{int(np.mean(heights))}')

## 10. Visualization — Sample Fundus Images

In [ ]:
DR_GRADE_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}

samples = train_df.sample(min(6, len(train_df)), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, (_, row) in enumerate(samples.iterrows()):
    img_bgr = cv2.imread(str(row['image_path']))
    if img_bgr is None: continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    grade = row['dr_grade']
    label = DR_GRADE_NAMES.get(grade, f'Grade {grade}')
    axes[i].set_title(f'{row["image_path"].name}\nDR: {label}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('DDR — Sample Fundus Images with DR Grade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save Processed Files to Google Drive
> Run this cell once after Cell 8 to persist `.npz` files across Colab sessions.
> Saved to: `My Drive/DRP_processed/ddr/`

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

drive_dest = Path('/content/drive/MyDrive/DRP_processed')
drive_dest.mkdir(parents=True, exist_ok=True)

src = processed_dir / DATASET_NAME
if src.exists():
    shutil.copytree(str(src), str(drive_dest / DATASET_NAME), dirs_exist_ok=True)
    npz_count = len(list((drive_dest / DATASET_NAME).rglob('*.npz')))
    print(f'Saved {npz_count} .npz files → {drive_dest / DATASET_NAME}')
else:
    print('No processed files found. Run Cell 8 first.')

## Next Steps — Notebook 02
- Load `.npz` files from `data/processed/ddr/` (or Google Drive) in Notebook 02
- Build a **DR grading classifier** (EfficientNet / ResNet) using `dr_grade` labels
- Or build a **lesion segmentation** model (U-Net) if switching to IDRiD dataset
- Metrics: **AUC-ROC**, **Quadratic Weighted Kappa**, **Accuracy** per DR grade
- Consider class-weighted loss due to grade imbalance in DDR